# Spark Runtime Architecture

{doc}`What Is Apache Spark? <01_what_is_apache_spark>` described Spark as an
engine that splits work across many machines. This page shows how that split
is organised: which processes exist, what each one does, and how your
Python code (running in a notebook or on your laptop) reaches them.

## The Building Blocks of a Spark Application

A running Spark program is called a **Spark application**. Every application
has the same three kinds of participants.

```mermaid
:caption: A classic Spark application. The driver plans, the executors do the work.
:align: center
:name: spark-classic-architecture

graph LR
    subgraph DriverProc[Driver]
        SS[SparkSession]
    end
    DriverProc <--> CM[Cluster manager]
    DriverProc --> E1
    DriverProc --> E2
    DriverProc --> E3
    subgraph W1[Worker node 1]
        E1[Executor]
    end
    subgraph W2[Worker node 2]
        E2[Executor]
    end
    subgraph W3[Worker node 3]
        E3[Executor]
    end
```

**The driver** is the process that runs your program's `main()` function and
coordinates the application
([Apache Spark: Cluster Mode Overview](https://spark.apache.org/docs/latest/cluster-overview.html)).
It:

- turns your DataFrame or SQL code into a logical plan, optimises it, and
  produces a physical plan;
- splits the work into **jobs**, **stages** and **tasks**: a job is the
  parallel computation spawned by an action such as `collect`, each job is
  divided into stages that depend on each other, and a task is the unit of
  work sent to one executor
  ([Apache Spark: Cluster Mode Overview](https://spark.apache.org/docs/latest/cluster-overview.html#glossary));
- schedules tasks onto executors and tracks their progress;
- collects results that you ask for explicitly (for example with `collect()`
  or `show()`).

**Executors** are processes on the worker nodes that run computations and
store data for your application. Each application gets its own executors,
which stay up for the whole application and run tasks in multiple threads
([Apache Spark: Cluster Mode Overview](https://spark.apache.org/docs/latest/cluster-overview.html)).
Each task is allocated one core by default (`spark.task.cpus = 1`), so an
executor runs as many tasks at once as it has cores
([Apache Spark: Configuration](https://spark.apache.org/docs/latest/configuration.html)).
These task-sized units of parallelism are often called **slots**: a cluster
with 4 executors of 8 cores each has 32 slots, so it can process 32 tasks at
once.

**The cluster manager** allocates resources and starts executors. Open-source
Spark supports Standalone, Hadoop YARN and Kubernetes cluster managers
([Apache Spark: Cluster Mode Overview](https://spark.apache.org/docs/latest/cluster-overview.html)).
On Databricks you never configure one yourself: you configure classic
compute, or use serverless compute, where Databricks automatically allocates
and manages the compute resources
([What is serverless compute?](https://docs.databricks.com/aws/en/compute/serverless/)).

:::{admonition} Note: The Driver Is a Single Point of Coordination
:class: note

There is exactly one driver per application. Anything you pull back to it has
to fit there: `toPandas()`, for example, collects every record of the
DataFrame to the driver program
([Convert between PySpark and pandas DataFrames](https://docs.databricks.com/aws/en/pandas/pyspark-pandas-conversion#example)).
Pulling a large distributed dataset back to the driver is one of the most
common ways to crash a Spark application.
:::

## From Code to Tasks

When you trigger a computation, the driver goes through the same pipeline
every time:

```mermaid
:caption: How a DataFrame query becomes tasks running on executors.
:align: center

graph LR
    A[DataFrame / SQL code] --> B[Unresolved logical plan]
    B --> C[Analyzed logical plan]
    C --> D[Optimized logical plan]
    D --> E[Physical plan]
    E --> F[Jobs, stages, tasks]
    F --> G[Executors]
```

1. **Analysis** resolves table and column names against the catalog (Unity
   Catalog on Databricks) and checks types. A typo in a column name fails
   here.
2. **Logical optimisation** (Catalyst) rewrites the plan, for example by
   pushing filters down towards the data source and dropping unused columns.
3. **Physical planning** picks concrete algorithms, such as which join
   strategy to use.
4. **Execution** cuts the physical plan into stages at every point where data
   must be redistributed between machines, and each stage into tasks.

`EXPLAIN EXTENDED` in SQL, or `df.explain(True)` in Python, prints the plans
from the first three steps: the parsed (unresolved) logical plan, the
analysed logical plan, the optimised logical plan and the physical plan
([EXPLAIN](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-qry-explain)).
{doc}`Spark and Databricks <../../04_monitoring_and_optimizing_spark_workloads_on_databricks/01_spark_and_databricks>`
shows how to read them in depth, and the example further down this page
prints them once you have a session. Jobs, stages, tasks and partitions get
their own treatment in
{doc}`Distributed Systems Programming Fundamentals <../03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`.
For now, the important point is that the driver plans and the executors
execute.

## Spark Classic and Spark Connect

The diagram above shows **Spark Classic**: your code runs *inside* the driver
process, together with the Spark engine. This ties your application tightly
to the cluster: an application that uses too much memory affects the driver
it shares with everyone else, and the application's dependencies can conflict
with the driver's.

**Spark Connect**, introduced in Apache Spark 3.4, decouples the two. The
client translates DataFrame operations into *unresolved* logical plans and
sends them to the server over gRPC. The server turns them into Spark's own
logical plans, runs the standard optimisation and execution process, and
streams results back as Apache Arrow-encoded row batches. Each application
then only affects its own process, and the server can be upgraded
independently of the applications
([Apache Spark: Spark Connect Overview](https://spark.apache.org/docs/latest/spark-connect-overview.html)).

```mermaid
:caption: Spark Connect. The client only builds plans; the server does all analysis and execution.
:align: center
:name: spark-connect-architecture

graph LR
    subgraph Client[Your machine or notebook]
        Code[Python code] --> Lib[Spark Connect client]
    end
    Lib -- "unresolved plan (gRPC)" --> Server
    Server -- "results (Arrow batches)" --> Lib
    subgraph Cluster[Spark cluster]
        Server[Spark Connect server / driver] --> Ex1[Executor]
        Server --> Ex2[Executor]
    end
```

## How It Works on Databricks

The DataFrame API looks identical in both modes. What changes is *when* work
happens
([Compare Spark Connect to Spark Classic](https://docs.databricks.com/aws/en/spark/connect-vs-classic#lazy-vs-eager)):

| Aspect | Spark Classic | Spark Connect |
| --- | --- | --- |
| Transformations and SQL queries | Lazy | Lazy |
| Schema analysis | Eager | Lazy, on the server |
| Schema access (`df.columns`, `df.schema`) | Local | One round trip to the server, then cached |
| Temporary views | Plan embedded at creation | Name looked up at execution |
| Python UDFs | Serialised at creation | Serialised at execution |

### Where Spark Connect Is Used on Databricks

Spark Connect is the protocol behind
([Compare Spark Connect to Spark Classic](https://docs.databricks.com/aws/en/spark/connect-vs-classic)):

- **Serverless compute**;
- **Databricks Connect**, the client library this course uses to run code
  from your IDE (see
  {doc}`Setting Up Databricks Connect <../../00_getting_started/01_setting_up_databricks_connect>`);
- Python notebooks on **standard** compute with Databricks Runtime 14.3 and
  above, and Scala notebooks on standard compute with Databricks Runtime 13.3
  and above;
- Lakeflow pipelines configured with an environment version.

In other words, every example in this course runs through Spark Connect. The
cell below starts the course's session, which
{ref}`Starting a Session <spark-session-start>` explains at the end of this
page.

In [1]:
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

Here are the plans for a small query on the `samples.nyctaxi.trips` table.
Compare the analysed plan, whose `Relation` still lists all six columns of
the table, with the physical plan at the bottom: its scan has a `ReadSchema`
of only the two columns the query needs, and the filter has been pushed into
the scan as `DataFilters`. On serverless compute the scan is a `PhotonScan`,
because Photon runs the query.

In [2]:
(
    spark.read.table("samples.nyctaxi.trips")
    .filter(F.col("fare_amount") > 50)
    .select("trip_distance", "fare_amount")
    .explain(True)
)

== Parsed Logical Plan ==
'Project ['trip_distance, 'fare_amount]
+- 'Filter '`>`('fare_amount, 50)
   +- 'UnresolvedRelation [samples, nyctaxi, trips], [], false

== Analyzed Logical Plan ==
trip_distance: double, fare_amount: double
Project [trip_distance#10394, fare_amount#10395]
+- Filter (fare_amount#10395 > cast(50 as double))
   +- SubqueryAlias samples.nyctaxi.trips
      +- Relation samples.nyctaxi.trips[tpep_pickup_datetime#10392,tpep_dropoff_datetime#10393,trip_distance#10394,fare_amount#10395,pickup_zip#10396,dropoff_zip#10397] parquet

== Optimized Logical Plan ==
Project [trip_distance#10394, fare_amount#10395]
+- Filter (isnotnull(fare_amount#10395) AND (fare_amount#10395 > 50.0))
   +- Relation samples.nyctaxi.trips[tpep_pickup_datetime#10392,tpep_dropoff_datetime#10393,trip_distance#10394,fare_amount#10395,pickup_zip#10396,dropoff_zip#10397] parquet

== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonScan parquet samples.nyctaxi.trips[trip_distanc

(spark-connect-practical-consequences)=
### Practical Consequences

Because analysis is deferred, **errors appear later** than you might expect.
Under Spark Connect the unresolved plan exists only on the client, so a
`filter` on a column that does not exist raises no error. The error appears
when something needs a resolved plan, such as `df.columns`, `df.schema` or an
action
([Compare Spark Connect to Spark Classic](https://docs.databricks.com/aws/en/spark/connect-vs-classic#trigger-eager-analysis-for-error-detection)).

The `samples.nyctaxi.trips` table has a `fare_amount` column but no
`tip_amount` column. This line runs without complaint:

In [3]:
trips = spark.read.table("samples.nyctaxi.trips")
bad = trips.filter("tip_amount > 5")  # no error yet: the plan is unresolved
print(type(bad).__name__)

DataFrame


Asking for the columns sends the plan to the server for analysis, and only
now does the typo fail:

In [4]:
bad.columns

{"ts": "2026-10-12 22:26:09.204", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `tip_amount` cannot be resolved. Did you mean one of the following? [`fare_amount`, `trip_distance`, `pickup_zip`, `dropoff_zip`, `tpep_pickup_datetime`]. SQLSTATE: 42703; line 1 pos 0;\n'Filter ('tip_amount > 5)\n+- SubqueryAlias samples.nyctaxi.trips\n   +- Relation samples.nyctaxi.trips[tpep_pickup_datetime#10411,tpep_dropoff_datetime#10412,trip_distance#10413,fare_amount#10414,pickup_zip#10415,dropoff_zip#10416] parquet\n\n\nJVM stacktrace:\norg.apache.spark.sql.catalyst.ExtendedAnalysisException\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.unresolvedAttributeError(QueryCompilationErrors.scala:942)\n\tat org.apache.spark.sql.catalyst.analysis.CheckAnalysis.org$apache$spark$sql$catalyst$analysis$CheckAnalysis$$failUnresolvedAttribute(CheckAnalysis.scala:301)\n\tat org.apache.spark.sql.catal

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `tip_amount` cannot be resolved. Did you mean one of the following? [`fare_amount`, `trip_distance`, `pickup_zip`, `dropoff_zip`, `tpep_pickup_datetime`]. SQLSTATE: 42703; line 1 pos 0;
'Filter ('tip_amount > 5)
+- SubqueryAlias samples.nyctaxi.trips
   +- Relation samples.nyctaxi.trips[tpep_pickup_datetime#10411,tpep_dropoff_datetime#10412,trip_distance#10413,fare_amount#10414,pickup_zip#10415,dropoff_zip#10416] parquet


JVM stacktrace:
org.apache.spark.sql.catalyst.ExtendedAnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.unresolvedAttributeError(QueryCompilationErrors.scala:942)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.org$apache$spark$sql$catalyst$analysis$CheckAnalysis$$failUnresolvedAttribute(CheckAnalysis.scala:301)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$16(CheckAnalysis.scala:708)
	at org.ap

The error is an `AnalysisException` with the error class
`UNRESOLVED_COLUMN.WITH_SUGGESTION`, and the message suggests the closest
existing columns, starting with `fare_amount`.
If you want a `try` block to catch analysis errors, put a schema access or an
action inside it.

Two other behaviours follow from the table above.

**Temporary views are looked up by name at execution time.** Under Spark
Connect a DataFrame created from a temporary view stores only the view's name,
so if you replace the view, the DataFrame reads the new data
([Compare Spark Connect to Spark Classic](https://docs.databricks.com/aws/en/spark/connect-vs-classic#create-unique-temporary-view-names)).
Watch the count change although `numbers` itself was never reassigned:

In [5]:
spark.createDataFrame([(1,)], "n INT").createOrReplaceTempView("arch02_numbers")
numbers = spark.table("arch02_numbers")
print("before:", numbers.count())

spark.createDataFrame([(1,), (2,), (3,)], "n INT").createOrReplaceTempView("arch02_numbers")
print("after: ", numbers.count())

before: 1


after:  3


**Python UDFs capture variables at execution time.** Under Spark Connect a
UDF is serialised when it runs, not when it is defined, so a later change to
a variable it uses changes its result. Databricks recommends wrapping the UDF
in a helper function that captures the value
([Compare Spark Connect to Spark Classic](https://docs.databricks.com/aws/en/spark/connect-vs-classic#wrap-udf-definitions)).
In the code below, the first UDF would pick up the reassigned `offset` and
add 100, while the one built by the factory keeps the value 1 it was created
with:

```python
offset = 1
add_offset = F.udf(lambda v: v + offset, "long")


def make_add(value):
    return F.udf(lambda v: v + value, "long")


add_one = make_add(offset)

offset = 100
spark.range(3).select(
    "id",
    add_offset("id").alias("late_binding"),  # 100, 101, 102
    add_one("id").alias("factory"),          # 1, 2, 3
).show()
```

:::{admonition} Unverified: Output of the UDF Example
:class: warning

This block is not executed on this page, because Python UDFs failed to start
on the course's Free Edition workspace when the page was written. The
expected values in the comments follow the behaviour described in
[Wrap UDF definitions](https://docs.databricks.com/aws/en/spark/connect-vs-classic#wrap-udf-definitions)
but haven't been confirmed by running the code. Run it yourself to check.
:::

## Databricks Compute Options

Databricks offers several kinds of compute that can run Spark
([Compute](https://docs.databricks.com/aws/en/compute/)):

- **Serverless compute**: Databricks automatically allocates and manages the
  compute resources, which speeds up start-up and minimises idle time
  ([What is serverless compute?](https://docs.databricks.com/aws/en/compute/serverless/)).
  It is the only compute available in Databricks Free Edition
  ([Free Edition limitations](https://docs.databricks.com/aws/en/getting-started/free-edition-limitations)),
  and it is what this course uses. Serverless supports only Spark Connect
  APIs, which brings some limits
  ([Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations)):
  - no RDD APIs;
  - DataFrame and SQL cache APIs (`df.cache()`, `df.persist()`,
    `CACHE TABLE`) raise an exception;
  - most Apache Spark compute configurations cannot be set;
  - global temporary views are not supported (session temporary views are);
  - there is no Spark UI; use the query profile instead;
  - ANSI mode is on by default, and the session time zone defaults to
    `Etc/UTC`.
- **Classic compute**: compute you configure yourself, including the
  Databricks Runtime version
  ([Compute configuration reference](https://docs.databricks.com/aws/en/compute/configure)).
  Its **access mode** decides who can use it: **standard** compute can be
  used by multiple users with data isolation among them, and **dedicated**
  compute is assigned to a single user or group
  ([Compute configuration reference](https://docs.databricks.com/aws/en/compute/configure#access-modes)).
  Standard compute does not support the RDD APIs; workloads that need them
  use dedicated compute
  ([Standard compute requirements and limitations](https://docs.databricks.com/aws/en/compute/access-mode-limitations#spark-api-limitations),
  [Dedicated compute overview](https://docs.databricks.com/aws/en/compute/dedicated-overview#when-to-use-dedicated-compute)).
- **SQL warehouses**: compute optimised for SQL queries, analytics and BI
  workloads
  ([Compute](https://docs.databricks.com/aws/en/compute/#sql-warehouses)). You
  won't use them for the DataFrame examples here.

Serverless notebooks and jobs run on a versioned **environment**, whose
versioned API stays compatible with newer versions. Environment version 6
ships Python 3.12.3 and Databricks Connect 19
([Serverless environment version 6](https://docs.databricks.com/aws/en/release-notes/serverless/environment-version/six)),
which is why this project pins `databricks-connect==19.1.*` in
`pyproject.toml`.

## When to Use Which Compute

| Compute | Choose it when | Trade-off |
| --- | --- | --- |
| Serverless | The default for notebooks and jobs, and the only option on Free Edition | No RDDs, no caching, few Spark configurations |
| Classic, standard access mode | Several users share a cluster and need a setting serverless doesn't allow | You size and manage it; no RDDs |
| Classic, dedicated | The workload needs RDD APIs, GPUs, R or privileged machine access | Only one user or group can use it |
| SQL warehouse | SQL and BI queries | Runs SQL only |

Databricks recommends standard access mode for classic compute unless the
functionality you need isn't supported
([Compute configuration reference](https://docs.databricks.com/aws/en/compute/configure#access-modes)),
and lists RDDs, GPUs, R and privileged machine access as the reasons to choose
dedicated compute
([Dedicated compute overview](https://docs.databricks.com/aws/en/compute/dedicated-overview#when-to-use-dedicated-compute)).

(spark-session-start)=
## Starting a Session

Everything starts with a `SparkSession`. In a Databricks notebook one already
exists as `spark`. From your IDE, create one with Databricks Connect:

```python
from databricks.connect import DatabricksSession

spark = (
    DatabricksSession.builder
    .serverless()
    .profile("free-edition")  # the CLI profile you created during setup
    .getOrCreate()
)
```

This repository wraps the same logic in `project.utils.sparksession`, which
falls back to a plain local `SparkSession` when Databricks Connect is not
installed. That is the cell at the top of every notebook in this course, and
the one this page ran
{ref}`above <spark-connect-practical-consequences>`.

## Common Mistakes

- **Expecting the error on the line with the typo.** As the
  {ref}`example above <spark-connect-practical-consequences>` showed, a bad
  column name in `filter` or `select` doesn't fail until something needs a
  resolved plan. Put `df.columns`, `df.schema` or an action inside the `try`
  block that should catch it
  ([Compare Spark Connect to Spark Classic](https://docs.databricks.com/aws/en/spark/connect-vs-classic#trigger-eager-analysis-for-error-detection)).
- **Calling `df.columns` on many new DataFrames in a loop.** Each new
  DataFrame's first schema access is a round trip to the server, so a loop
  that adds a column and then checks `df.columns` on every iteration makes one
  request per iteration. Keep track of the column names yourself, for example
  in a Python set
  ([Compare Spark Connect to Spark Classic](https://docs.databricks.com/aws/en/spark/connect-vs-classic#avoid-too-many-eager-analysis-requests)).
- **Reusing temporary view names.** As shown above, replacing a view silently
  changes every DataFrame that was built from it. Give temporary views unique
  names, for example with a UUID in the name
  ([Compare Spark Connect to Spark Classic](https://docs.databricks.com/aws/en/spark/connect-vs-classic#create-unique-temporary-view-names)).
- **Collecting large results to the driver.** `collect()` and `toPandas()`
  bring every row back to the single driver. Aggregate or filter first, and
  only collect small results
  ([Convert between PySpark and pandas DataFrames](https://docs.databricks.com/aws/en/pandas/pyspark-pandas-conversion#example)).

With a session in hand, the next step is the main data abstraction you will
use with it:
{doc}`Introduction to DataFrames <../02_spark_dataframes_and_sql/01_introduction_to_dataframes>`.

## Resources

- [Cluster mode overview (Apache Spark)](https://spark.apache.org/docs/latest/cluster-overview.html)
- [Spark Connect overview (Apache Spark)](https://spark.apache.org/docs/latest/spark-connect-overview.html)
- [Compare Spark Connect to Spark Classic (Databricks)](https://docs.databricks.com/aws/en/spark/connect-vs-classic)
- [Serverless compute limitations (Databricks)](https://docs.databricks.com/aws/en/compute/serverless/limitations)
- [Serverless environment versions (Databricks)](https://docs.databricks.com/aws/en/release-notes/serverless/environment-version/)